In [ ]:
import albumentations as A 
import cv2 
import numpy as np
import os
import pandas as pd 
import torch

from collections import defaultdict
from importlib import import_module
from sklearn.metrics import f1_score, recall_score, roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from skp.toolbox.functions import load_model_from_config

In [ ]:
def pad_to_aspect_ratio(img: np.ndarray, aspect_ratio: float) -> np.ndarray:
    """
    Pads to specified aspect ratio, only if current aspect ratio is
    greater.
    """
    h, w = img.shape[:2]
    if h / w > aspect_ratio:
        new_w = round(h / aspect_ratio)
        w_diff = new_w - w
        left_pad = w_diff // 2
        right_pad = w_diff - left_pad
        padding = ((0, 0), (left_pad, right_pad))
        if img.ndim == 3:
            padding = padding + ((0, 0),)
        img = np.pad(img, padding, mode="constant", constant_values=0)
    return img

In [ ]:
class MammoDataset(Dataset):

    def __init__(self, df):
        self.df = df.copy()
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(os.path.join("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/train_cropped_png/", row.filename), 0)
        # 2048 x 1024 
        img1 = pad_to_aspect_ratio(img.copy(), 2048 / 1024)
        img1 = A.Resize(2048, 1024)(image=img1)["image"]
        # 1920 x 1280
        img2 = pad_to_aspect_ratio(img.copy(), 1920 / 1280)
        img2 = A.Resize(1920, 1280)(image=img2)["image"]
        # 1536 x 1536
        img3 = A.Compose([A.LongestMaxSize(1536), A.PadIfNeeded(1536, 1536)])(image=img.copy())["image"]
        return {
            "img1": torch.from_numpy(img1).unsqueeze(0),
            "img2": torch.from_numpy(img2).unsqueeze(0),
            "img3": torch.from_numpy(img3).unsqueeze(0),
            "label": torch.tensor(row.cancer),
            "patient_id": row.patient_id,
            "image_id": row.image_id, 
            "laterality": row.laterality,
            "view": row["view"],
            "breast_id": row.breast_id,
            "filename": row.filename,
        }

In [ ]:
def run_test(df, model_list):
    pred_df = defaultdict(list)
    dataset = MammoDataset(df)
    loader = DataLoader(dataset, batch_size=32, num_workers=8, shuffle=False, drop_last=False)
    for batch in tqdm(loader, total=len(loader)):
        with torch.inference_mode():
            for i, (each_img, each_model) in enumerate(zip([f"img{_ii + 1}" for _ii in range(len(model_list))], model_list)):
                x = batch[each_img]
                with torch.inference_mode():
                    out = each_model({"x": x.to("cuda:0")})["logits"].sigmoid()[:, 0]
                    pred_df[f"y_pred{i}"].extend(list(out.cpu().numpy()))
        pred_df["y_true"].extend(list(batch["label"].numpy()))
        pred_df["patient_id"].extend(list(batch["patient_id"].numpy()))
        pred_df["image_id"].extend(list(batch["image_id"].numpy()))
        pred_df["laterality"].extend(list(batch["laterality"]))
        pred_df["view"].extend(list(batch["view"]))
        pred_df["breast_id"].extend(list(batch["breast_id"]))
        pred_df["filename"].extend(list(batch["filename"]))
    return pd.DataFrame(pred_df)

## Split 1

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/6a3b30f7/fold0/checkpoints/ema_weights.pt"
model1 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v02"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/ddfeb3f6/fold1/checkpoints/ema_weights.pt"
model2 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v04"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/b386692c/fold2/checkpoints/ema_weights.pt"
model3 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

In [ ]:
test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v01.csv")
test_df

In [ ]:
pred_df1 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df1["y_pred3"] = (pred_df1.y_pred0 + pred_df1.y_pred1 + pred_df1.y_pred2) / 3
for i in range(4):
    auc = roc_auc_score(pred_df1.y_true.values, pred_df1[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df1_grouped = pred_df1.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4):
    auc = roc_auc_score(pred_df1_grouped.y_true.values, pred_df1_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

In [ ]:
thresholds = np.arange(0.05, 1, 0.05)

for th in thresholds:
    print(f"F1 @ {th:0.2f}: {f1_score(pred_df1_grouped.y_true.values, pred_df1_grouped.y_pred3.values >= th):0.4f}")

for th in thresholds:
    print(f"Sensitivity @ {th:0.2f}: {recall_score(pred_df1_grouped.y_true.values, pred_df1_grouped.y_pred3.values >= th):0.4f}")

## Split 2

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/c55fbaf7/fold0/checkpoints/ema_weights.pt"
model1 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v02"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/fa6df1c9/fold1/checkpoints/ema_weights.pt"
model2 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v04"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/cbbc1071/fold2/checkpoints/ema_weights.pt"
model3 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v02.csv")

pred_df2 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df2["y_pred3"] = (pred_df2.y_pred0 + pred_df2.y_pred1 + pred_df2.y_pred2) / 3
for i in range(4):
    auc = roc_auc_score(pred_df2.y_true.values, pred_df2[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df2_grouped = pred_df2.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4): 
    auc = roc_auc_score(pred_df2_grouped.y_true.values, pred_df2_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

## Split 3

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/6b00696a/fold0/checkpoints/ema_weights.pt"
model1 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v02"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/387cf1fd/fold1/checkpoints/ema_weights.pt"
model2 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v04"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/9c9a8feb/fold2/checkpoints/ema_weights.pt"
model3 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v03.csv")

pred_df3 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df3["y_pred3"] = (pred_df3.y_pred0 + pred_df3.y_pred1 + pred_df3.y_pred2) / 3
for i in range(4):
    auc = roc_auc_score(pred_df3.y_true.values, pred_df3[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df3_grouped = pred_df3.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4): 
    auc = roc_auc_score(pred_df3_grouped.y_true.values, pred_df3_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

In [ ]:
auc_list = []
for tmp_df in [pred_df1_grouped, pred_df2_grouped, pred_df3_grouped]:
    auc = roc_auc_score(tmp_df.y_true.values, tmp_df.y_pred3.values)
    auc_list.append(auc)

print(f"Mean AUC: {np.mean(auc_list):0.4f}")
print(f"Std AUC: {np.std(auc_list):0.4f}")

In [ ]:
for tmp_df in [pred_df1_grouped, pred_df2_grouped, pred_df3_grouped]:
    fpr, tpr, thresholds = roc_curve(tmp_df.y_true.values, tmp_df.y_pred3.values)
    # 100% sensitivity
    idx = np.argmin(np.abs(1 - tpr))
    print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.3f}, Threshold: {thresholds[idx]:0.4f}")
    # 98% sensitivity
    idx = np.argmin(np.abs(0.98 - tpr))
    print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.3f}, Threshold: {thresholds[idx]:0.4f}")
    # 95% sensitivity
    idx = np.argmin(np.abs(0.95 - tpr))
    print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.3f}, Threshold: {thresholds[idx]:0.4f}")
    # 90% sensitivity
    idx = np.argmin(np.abs(0.90 - tpr))
    print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.3f}, Threshold: {thresholds[idx]:0.4f}")
    print("\n")
    

In [ ]:
a = 0.0211
b = 0.0148
c = 0.0194

np.mean([a, b, c]), np.std([a, b, c])

In [ ]:
(0.907 + 0.904 + 0.904) / 3

In [ ]:
pred_df1_grouped.sort_values("y_pred3", ascending=False)